In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_talent_risk_events
COMMENT "Raw streaming audit events ingested via Auto Loader"
AS SELECT * FROM STREAM read_files("/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit", format=> 'json', inferColumnTypes => true);

CREATE OR REFRESH STREAMING TABLE silver_talent_risk_events (
-- Aturan 1: DROP RECORD jika melanggar
    CONSTRAINT valid_audit_id 
        EXPECT (audit_id IS NOT NULL) 
        ON VIOLATION DROP ROW,
        
    CONSTRAINT valid_employee_id 
        EXPECT (employee_id IS NOT NULL AND employee_id > 0) 
        ON VIOLATION DROP ROW,

    -- Aturan 2: TRACK / WARN (pertahankan baris, catat metrik pelanggaran di UI)
    CONSTRAINT risk_score_in_benchmark 
        EXPECT (risk_score <= 100),
        
    CONSTRAINT valid_flag_reason 
        EXPECT (flag_reason IS NOT NULL)
)
COMMENT "Cleaned silver streaming table with data quality expectations"
AS SELECT 
    audit_id,
    employee_id,
    flag_reason,
    risk_score,
    current_timestamp() AS silver_processed_at
FROM STREAM(LIVE.bronze_talent_risk_events);

CREATE MATERIALIZED VIEW gold_critical_risk_escalations (
    -- Hentikan eksekusi pipeline jika terjadi anomali integritas data kritis
    CONSTRAINT valid_critical_threshold 
        EXPECT (peak_risk_score >= 0 AND peak_risk_score <= 100) 
        ON VIOLATION FAIL UPDATE
)
COMMENT "High-priority risk escalations requiring guaranteed data integrity"
TBLPROPERTIES ("quality" = "gold")
AS SELECT 
    employee_id,
    COUNT(audit_id) AS total_audits,
    MAX(risk_score) AS peak_risk_score
FROM LIVE.silver_talent_risk_events
GROUP BY employee_id
HAVING MAX(risk_score) >= 80;

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5685452990588889>, line 6
      1 # ==============================================================================
      2 # DELTA LIVE TABLES (DLT) PIPELINE: DATA QUALITY EXPECTATIONS
      3 # USE CASE: TALENT ASSET & RISK AUDIT GOVERNANCE
      4 # ==============================================================================
----> 6 import dlt
      7 from pyspark.sql import functions as F
      9 # --- 1. BRONZE LAYER: INGESTION WITH AUTO LOADER ---
     10 # Ingestion awal dari Volume storage tanpa ekspektasi ketat

ModuleNotFoundError: No module named 'dlt'